# Phase 2: Every gas from all sensors, nominal + cross-sensitivity edges (Issue #17)

**Question:** Given hour, weather, and all five sensors, how well can we recover the true level of each gas, and does modelling cross-sensitivity help?

1. **Per gas:** how much do sensors + weather add over `HourOnlyBaseline`?
2. **Cross edges:** nominal-only vs nominal + cross. Do the dashed edges improve accuracy, calibration (log-likelihood) and concentration error?
3. **Overconfidence:** Phase 1b (all sensors, CO only) was more accurate but worse calibrated. Dropping any sensor raised log-likelihood, because the sensors were treated as independent witnesses of CO. With the other gases in the network, does that go away? Rerun the drop-one-sensor ablation.
4. **EM:** each gas reference is missing on ~18% of hours (NMHC ~90%). How much does EM gain over complete-case counting?

**Network** (Figure 2; cross edges from the cross-sensitivity study #16):

```
hour ──► CO, NMHC, NOx, NO2
gas  ──► its nominal sensor             S1 ← CO,  S2 ← NMHC,  S3 ← NOx,  S4 ← NO2
gas  ··► cross-sensitive sensors        S4 ← CO, NOx;  S5 ← CO, NOx
W = (T, AH) ──► every sensor
```

**Assumptions:** 
- **A1.** Each gas depends on hour only. The gases are independent given hour; their co-movement beyond the hour is not modelled.
- **A2.** Each sensor depends only on its gas parents and weather. Given those, the sensors are independent of each other.
- **A3.** Missingness is ignorable given the observed values (MAR).
- **A4.** Hours are independent given the CPTs (no temporal dependence; that's Phase 4).

**Rules.** Every choice (edges, bins) is made on **val**. **Test** is scored once, at the end, for the chosen configuration. Every metric for a (model, gas, evidence) comes from **one** `predict_proba` call.

**Sections**
1. Setup
2. Data and variables
3. Edges and networks
4. Splits
5. Metrics
6. Grid: edges × seed (6 runs, 3 bins)
7. Results per gas, nominal vs cross, and the chosen configuration
8. Calibration, confusion matrices, and do the CPTs match the study?
9. Follow-ups: drop-one-sensor ablation, EM vs complete-case
10. Assumptions A1–A4
11. Drift check (time-based split)
12. Test set (chosen configuration, once)

**Outputs** (outputs are stripped on commit, so the numbers live in files): the grid in `notebooks/experiments/phase2_grid.csv`, and figures and follow-up tables in `figures/phase2/`.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cse250a.modules.config import config
from cse250a.modules.config.paths import REPO_ROOT
from cse250a.modules.core import TIME_CARDS, Gas, Network, Sensor, Time, Weather
from cse250a.modules.data import Discretizer, blocked_split, load_air_quality
from cse250a.modules.eval import partial_spearman, score
from cse250a.modules.models import DiscreteBN, HourOnlyBaseline

## 2. Data and variables

- **Gases:** CO, NMHC, NOx, NO2.
- **NMHC** is scored **only on hours with a real NMHC reading**, always with `n`.
- **Weather `W` = T + AH**.
- **Sensors:** all five, S1–S5.
- **Evidence** for every query: hour, weather and the five sensors.

Rows with a missing hour or weather are dropped, because those are inputs with no CPT, so they must be observed on every row. The sensors are missing on exactly those rows too, so after this every sensor is always observed and only the gas references have holes.

In [ ]:
df = load_air_quality( config.paths.data_dir / "air_quality" / "AirQualityUCI.csv")
print(df.shape)
df.isna().mean().round(3).rename("fraction missing").to_frame().T

In [ ]:
gases = (Gas.CO, Gas.NMHC, Gas.NOX, Gas.NO2)
weather = (Weather.T, Weather.AH)
sensors = tuple(Sensor)

before = len(df)
df = df.dropna(subset=[Time.HOUR, *weather])
print(f"dropped {before - len(df)} rows with missing weather; {len(df)} left")

**Two things to watch in the results:**
- **CO vs Phase 1.** S2 is tied only to NMHC, and the gases are independent given the hour (A1), so S2 no longer helps predict CO. Phase 2's CO accuracy may come out lower than Phase 1's; that is expected, and called out where it happens.
- **NMHC may partly mean "traffic".** S2 tracks all the traffic gases about equally, so NMHC predictions may partly reflect traffic pollution in general rather than NMHC specifically (a limitation for the report).

Section 4 lists how many hours each split has an NMHC reading for; those are the hours NMHC is scored on.

## 3. Edges and networks

From the cross-sensitivity study (#16, `docs/decisions.md`). A gas has a cross edge only if its partial Spearman ρ with the sensor (controlling for the nominal gas, T, AH and hour) and its held-out CPT log-likelihood gain were both clearly non-zero and consistent across seeds.

The two edge sets compared in the grid:
- **nominal:** each sensor's own gas: S1 ← CO, S2 ← NMHC, S3 ← NOx, S4 ← NO2. S5 has no nominal gas (there's no O3 reference), so here it has weather parents only and carries no gas information.
- **nominal+cross:** adds the cross-sensitivities S4 ← CO, NOx (Muska's finding) and S5 ← CO, NOx (what S5 responded to best).

This matches Figure 2.

In [ ]:
nominal = {
    Sensor.S1: (Gas.CO,),
    Sensor.S2: (Gas.NMHC,),
    Sensor.S3: (Gas.NOX,),
    Sensor.S4: (Gas.NO2,),
}
cross = {
    Sensor.S4: (Gas.CO, Gas.NOX),
    Sensor.S5: (Gas.CO, Gas.NOX),
}
edge_sets = {
    "nominal": nominal,
    "nominal+cross": {s: (*nominal.get(s, ()), *cross.get(s, ())) for s in sensors},
}


def structure(edges, weather=weather):
    """hour → every gas; each sensor ← its gas parents + weather."""
    return {
        **{g: (Time.HOUR,) for g in gases},
        **{s: (*edges.get(s, ()), *weather) for s in sensors},
    }


def phase1b_structure(weather=weather, target=Gas.CO):
    """Phase 1b's network (example.ipynb): one gas, every sensor its child."""
    return {
        target: (Time.HOUR,),
        **{s: (target, *weather) for s in sensors},
    }

## 4. Splits

`blocked_split` shuffles whole **weeks** into train / val / test (50 / 25 / 25), so neighbouring hours don't land on both sides. Bin edges are learned on **train only**, then applied to every part. Each split also keeps each gas's **bin means** c̄ₖ (the mean raw concentration of the training hours in bin k), which turn a posterior back into a concentration:

$$\widehat{c}_t = \sum_k P(\text{bin}_k \mid \text{evidence}_t)\,\bar c_k$$

`time_split` (section 11) is the chronological alternative: train on the earliest half, val and test on later months.

In [ ]:
n_bins = 3
seeds = (0, 1, 2)

train_raw, val_raw, test_raw = blocked_split(df, seed=config.seed)
print({name: len(d) for name, d in [("train", train_raw), ("val", val_raw), ("test", test_raw)]})

In [ ]:
binned = [*gases, *sensors, *weather]
disc = Discretizer(n_bins=n_bins).fit(train_raw, binned)
train, val, test = (disc.transform(d) for d in (train_raw, val_raw, test_raw))

cards = {**disc.cards, **TIME_CARDS}
print("cards:", {str(k): v for k, v in cards.items() if k in binned or k == Time.HOUR})
train[binned].head()

In [ ]:
def time_split(df, train=0.5, val=0.25, seed=None):
    """Chronological 50 / 25 / 25: train on the earliest hours, test on the latest.
    `seed` is unused; it keeps the signature of blocked_split."""
    a, b = round(len(df) * train), round(len(df) * (train + val))
    return df.iloc[:a], df.iloc[a:b], df.iloc[b:]


def make_split(seed, *, split=blocked_split):
    """The two cells above for any seed, plus each gas's bin means c̄ₖ (from train)."""
    train_raw, val_raw, test_raw = split(df, seed=seed)
    binned = [*gases, *sensors, *weather]
    disc = Discretizer(n_bins=n_bins).fit(train_raw, binned)
    train, val, test = (disc.transform(d) for d in (train_raw, val_raw, test_raw))
    cards = {**disc.cards, **TIME_CARDS}
    c_bar = {g: train_raw[g].groupby(train[g]).mean() for g in gases}  # index: bin → mean
    return {"train_raw": train_raw, "val_raw": val_raw, "test_raw": test_raw,
            "train": train, "val": val, "test": test, "cards": cards, "c_bar": c_bar}


splits = {seed: make_split(seed) for seed in seeds}
pd.DataFrame({
    f"seed {seed}": {
        **{f"{k} hours": len(sp[f"{k}_raw"]) for k in ("train", "val", "test")},
        **{f"{k} NMHC hours": int(sp[f"{k}_raw"][Gas.NMHC].notna().sum()) for k in ("train", "val", "test")},
    }
    for seed, sp in splits.items()
})

In [ ]:
for name, edges in edge_sets.items():
    network = Network(
        structure=structure(edges),
        cards=cards,
    ).validate(train.columns)
    print(f"── {name}")
    print(network)

## 5. Metrics

Per gas, on the hours where that gas's reference is observed (`n`):
- **accuracy**: the most likely bin is the true bin
- **within_one**: the most likely bin is at most one bin away (bins are ordered, so low↔high is worse than low↔medium)
- **macro_f1**: one-vs-rest F1 averaged over bins
- **bere**: one-vs-rest balanced error rate, averaged over bins; lower is better
- **mean_log_likelihood**: mean log P(true bin). The best single number for a probabilistic model; higher (closer to 0) is better
- **ECE**: expected calibration error. Hours are grouped by confidence (the top probability); ECE is the hour-weighted mean of |hit rate − confidence|. 0 is perfectly calibrated
- **MAE / RMSE**: of the concentration estimate Σₖ P(binₖ) c̄ₖ, in the gas's units
- **floor_MAE / floor_RMSE**: the **resolution floor**, the error of a *perfect* bin prediction that puts every hour at its true bin's c̄ₖ. It's the practical limit of these bins; a soft posterior can land slightly below it, because averaging bin means sometimes beats the hard bin mean

Raw scores fall as bins increase (more bins are harder to hit), so bin counts are compared by **gain over `HourOnlyBaseline`**, which gets the same split, bins and evidence.

In [ ]:
evidence = (Time.HOUR, *weather, *sensors)
metrics = ["n", "accuracy", "macro_f1", "bere", "mean_log_likelihood"]
quality = ["accuracy", "within_one", "macro_f1", "mean_log_likelihood"]  # higher is better
units = {"CO": "mg/m³", "NMHC": "µg/m³", "NOx": "ppb", "NO2": "µg/m³"}
FIG = "phase2"  # figures and follow-up tables go to figures/phase2/


def short(name):
    """'PT08.S4(NO2)' -> 'S4', 'NOx(GT)' -> 'NOx'."""
    if name.startswith("PT08."):
        return name[5:7]
    return name.split("(")[0]


def reliability(proba, truth, bins=10):
    """Confidence vs hit rate, grouped by confidence. Returns (table, ECE)."""
    seen = truth.notna()
    conf = proba[seen].max(axis=1)
    hit = (proba[seen].idxmax(axis=1) == truth[seen]).astype(float)
    bucket = np.minimum((conf * bins).astype(int), bins - 1)
    t = pd.DataFrame({"confidence": conf, "hit_rate": hit}).groupby(bucket).agg(
        confidence=("confidence", "mean"), hit_rate=("hit_rate", "mean"), n=("hit_rate", "size")
    )
    ece = float((t.n / t.n.sum() * (t.hit_rate - t.confidence).abs()).sum())
    return t, ece


def gas_metrics(proba, split, gas, part="val"):
    """Every metric for one gas from one predict_proba output: score() (what
    Model.evaluate returns, on the hours with a reference), plus within-one-bin,
    ECE, and the concentration error from example.ipynb section 10."""
    truth = split[part][gas]
    seen = truth.notna()
    s = score(truth[seen], proba[seen])

    pred = proba.idxmax(axis=1)
    off = (pred[seen] - truth[seen]).abs()

    c_bar = split["c_bar"][gas]
    estimate = pd.Series(proba.to_numpy() @ c_bar.loc[proba.columns].to_numpy(), index=proba.index)
    raw = split[f"{part}_raw"][gas]
    err = estimate[seen] - raw[seen]
    floor = truth[seen].map(c_bar) - raw[seen]  # a perfect bin prediction

    return {
        **{k: s[k] for k in metrics},
        "within_one": float((off <= 1).mean()),
        "ECE": reliability(proba, truth)[1],
        "MAE": err.abs().mean(),
        "RMSE": np.sqrt((err ** 2).mean()),
        "floor_MAE": floor.abs().mean(),
        "floor_RMSE": np.sqrt((floor ** 2).mean()),
    }


def evaluate_all(model, split, part="val", gases=gases, evidence=evidence):
    """predict_proba once per gas. Returns ({gas: proba}, [one metrics row per gas])."""
    probas, rows = {}, []
    for gas in gases:
        probas[gas] = model.predict_proba(split[part], gas, evidence)
        rows.append({"gas": short(gas), "model": model.name, **gas_metrics(probas[gas], split, gas, part)})
    return probas, rows


def fit_bn(split, struct, **kwargs):
    network = Network(
        structure=struct,
        cards=split["cards"],
    ).validate(split["train"].columns)
    return DiscreteBN(network, **kwargs).fit(split["train"])


def mean_std(frame, by, cols, digits=3):
    """'mean ± std' over seeds, with n as its mean."""
    g = frame.groupby(by, sort=False)
    out = g[cols].mean().round(digits).astype(str) + " ± " + g[cols].std().fillna(0).round(digits).astype(str)
    if "n" in frame:
        out.insert(0, "n", g["n"].mean().round().astype(int))
    return out

## 6. Grid: edges × seed

edges ∈ {nominal only, nominal+cross} × seed ∈ {0, 1, 2} at 3 bins → 6 runs. Each run fits **one** BN (EM, `max_iter=100`) and answers every gas from it, and fits `HourOnlyBaseline` on the same split and bins. Every `predict_proba` is kept in `posteriors`, so later sections reuse it instead of predicting again.


In [ ]:
rows, fits, posteriors = [], {}, {}
for name, edges in edge_sets.items():
    for seed in seeds:
        sp = splits[seed]
        t0 = time.perf_counter()
        bn = fit_bn(sp, structure(edges))
        hour = HourOnlyBaseline(cards=sp["cards"]).fit(sp["train"])
        fits[(name, seed)] = bn

        r = bn.result
        for model in (bn, hour):
            posteriors[(name, seed, model.name)], gas_rows = evaluate_all(model, sp)
            rows += [{"edges": name, "seed": seed, **g} for g in gas_rows]
        print(f"{name:14} seed={seed}: {r.n_iter} iterations, converged: {r.converged}, "
              f"monotone: {r.is_monotone()}, {time.perf_counter() - t0:.0f}s", flush=True)

grid = pd.DataFrame(rows)
grid.to_csv(REPO_ROOT / "notebooks" / "experiments" / "phase2_grid.csv", index=False)
grid.head()

## 7. Results

### 7.1 Per gas: Phase 2 vs `HourOnlyBaseline`

`gain_*` = BN − baseline on the same run (positive is better). `MAE_hour` is the baseline's concentration error and `floor_MAE` the resolution floor, both in the gas's units. Mean ± std over seeds 0–2.

In [ ]:
keys = ["edges", "seed", "gas"]
bn_rows = grid[grid.model == "discrete_bn"].set_index(keys)
hour_rows = grid[grid.model == "hour_baseline"].set_index(keys)

gains = bn_rows.copy()
for m in quality:
    gains[f"gain_{m}"] = bn_rows[m] - hour_rows[m]
gains["ECE_hour"] = hour_rows["ECE"]
gains["MAE_hour"] = hour_rows["MAE"]
gains["rel_MAE"] = bn_rows["MAE"] / hour_rows["MAE"]  # < 1: better than the hour alone
gains = gains.reset_index()

per_gas = mean_std(
    gains,
    ["gas", "edges"],
    ["accuracy", "gain_accuracy", "within_one", "gain_within_one", "macro_f1", "gain_macro_f1",
     "bere", "mean_log_likelihood", "gain_mean_log_likelihood", "ECE", "ECE_hour",
     "MAE", "MAE_hour", "floor_MAE", "RMSE"],
)
per_gas.to_csv(config.paths.figure(FIG, "per_gas.csv"))
per_gas

In [ ]:
order = ["hour baseline", "nominal", "nominal+cross", "resolution floor"]
base = gains[gains.edges == "nominal"]
long = pd.concat([
    gains.assign(label=gains.edges)[["gas", "seed", "label", "MAE"]],
    base.assign(label="hour baseline", MAE=base.MAE_hour)[["gas", "seed", "label", "MAE"]],
    base.assign(label="resolution floor", MAE=base.floor_MAE)[["gas", "seed", "label", "MAE"]],
])

fig, axes = plt.subplots(1, len(gases), figsize=(15, 3.6))
for ax, gas in zip(axes, map(short, gases), strict=True):
    sns.barplot(data=long[long.gas == gas], x="label", y="MAE", order=order, errorbar="sd", ax=ax)
    ax.set_title(gas)
    ax.set_xlabel("")
    ax.set_ylabel(f"MAE ({units[gas]})")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("Concentration MAE on val (mean ± std over seeds)", y=1.03)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "mae_vs_floor.png"), bbox_inches="tight")
plt.show()

**Reading:** (val, nominal+cross, mean ± std over seeds; NMHC scored only on hours with a real reading)
- **CO and NOx clearly beat the hour baseline.** CO: accuracy 0.773 vs 0.586 (n = 1853), mean log-lik −0.570 vs −0.899, MAE 0.544 vs 0.857 mg/m³ (floor 0.461). NOx: accuracy 0.709 vs 0.495 (n = 1855), MAE 89.5 vs 136.5 ppb (floor 74.1).
- **NMHC beats it on average, with a very wide spread:** accuracy 0.732 ± 0.113 vs 0.613 (gain 0.119 ± 0.127), MAE 88.9 ± 20.1 vs 109.9 µg/m³, on only n ≈ 180 hours per seed (167 / 205 / 168). Seed 0 is much worse (accuracy 0.611, log-lik −1.047, ECE 0.25) than seeds 1 and 2 (accuracy 0.834 and 0.750). As expected for a gas with so few observed hours, a single split can't be trusted for NMHC.
- **NO2 barely beats it.** With nominal edges only it doesn't (accuracy gain 0.002 ± 0.027, log-lik gain −0.010 ± 0.014). With the cross edges: log-lik +0.034 ± 0.009, MAE 31.1 vs 32.0 µg/m³. S4 mostly tracks CO (section 8.3), so NO2 has no good witness, and its MAE stays far above the floor (31 vs 17).
- **CO is lower than Phase 1, as anticipated.** Seed 0, 3 bins: Phase 2 has better log-likelihood (−0.541 vs −0.662) but lower accuracy (0.776 vs 0.828) and slightly worse MAE (0.580 vs 0.556). With S2 tied only to NMHC, and the gases independent given the hour (A1), S2 no longer says anything about CO. The ablation (9.1) confirms it: in Phase 2, dropping S2 changes CO by exactly 0, while in Phase 1b it costs 0.032 accuracy.

### 7.2 Nominal-only vs nominal + cross

Paired by split: (nominal+cross) − (nominal) on the same seed and bins. Positive is better for the quality metrics; **negative** is better for ECE, MAE and RMSE.

In [ ]:
diff_cols = [*quality, "ECE", "MAE", "RMSE"]
by_edges = grid[grid.model == "discrete_bn"].set_index(keys)
cross_vs_nominal = (by_edges.loc["nominal+cross", diff_cols] - by_edges.loc["nominal", diff_cols]).reset_index()
cross_table = mean_std(cross_vs_nominal, ["gas"], diff_cols)
cross_table.to_csv(config.paths.figure(FIG, "cross_vs_nominal.csv"))
cross_table

**Reading:**
- **The cross edges help CO, NOx and NO2 on accuracy, log-likelihood and concentration error.** CO: accuracy +0.028 ± 0.010, log-lik +0.028 ± 0.023, MAE −0.046 ± 0.019 mg/m³. NOx: accuracy +0.042 ± 0.015, log-lik +0.054 ± 0.025, MAE −5.8 ± 3.2 ppb. NO2: log-lik +0.044 ± 0.023, accuracy +0.015 ± 0.028 (within noise).
- NMHC is unchanged (no cross edge touches it), as expected.
- **Calibration is mixed, not better.** ECE improves for NO2 (−0.046 ± 0.017) but gets worse for CO (+0.045 ± 0.013) and NOx (+0.015 ± 0.006). So "cross edges help log-likelihood and calibration more than accuracy" holds only partly: log-lik gains are about the size of the accuracy gains, and CO becomes more overconfident (section 8.1). S4 and S5 are extra CO witnesses that share unmodelled noise with S1 (A2).

### 7.3 Chosen configuration

**Selection criterion:** concentration MAE relative to the hour baseline (`rel_MAE` = MAE / MAE_hour, so gases in different units can be averaged), averaged over CO, NOx and NO2 and over seeds, on **val**. Lower is better. NMHC is reported but left out of the average: it is scored on only ~100–200 val hours per seed, and no cross edge touches it, so it can't tell the two edge sets apart.

In [ ]:
select_gases = ["CO", "NOx", "NO2"]
selection = (
    gains[gains.gas.isin(select_gases)]
    .groupby(["edges", "seed"])["rel_MAE"].mean()  # one number per run
    .groupby("edges").agg(["mean", "std"])
    .sort_values("mean")
)
chosen = selection["mean"].idxmin()
print(f"chosen: edges={chosen}, bins={n_bins}")
selection.round(4)

## 8. Calibration, confusion, and the learned CPTs

### 8.1 Calibration: confidence vs hit rate

For the chosen configuration, pooled over seeds. Each point is a group of val hours with similar confidence (the model's top probability); on the diagonal, a model that says 80% is right 80% of the time. Below the diagonal is **overconfident**.

In [ ]:
def pooled(model_name, gas, edges=chosen):
    """(proba, truth) for one gas, concatenated over seeds."""
    probas = [posteriors[(edges, s, model_name)][gas] for s in seeds]
    truths = [splits[s]["val"][gas] for s in seeds]
    return pd.concat(probas, ignore_index=True), pd.concat(truths, ignore_index=True)


fig, axes = plt.subplots(1, len(gases), figsize=(15, 3.6), sharey=True)
for ax, gas in zip(axes, gases, strict=True):
    for model_name, label in [("discrete_bn", "Phase 2 BN"), ("hour_baseline", "hour baseline")]:
        t, ece = reliability(*pooled(model_name, gas))
        ax.plot(t.confidence, t.hit_rate, marker="o", label=f"{label} (ECE {ece:.3f})")
    ax.plot([0, 1], [0, 1], ls="--", c="gray", lw=1)
    ax.set_title(short(gas))
    ax.set_xlabel("confidence")
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel("hit rate")
fig.suptitle(f"Calibration on val, {chosen}, {n_bins} bins (pooled over seeds)", y=1.03)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "calibration.png"), bbox_inches="tight")
plt.show()

### 8.2 Confusion matrices

Chosen configuration, pooled over seeds, each row normalized to sum to 1 (row = true bin). Mass far from the diagonal (low ↔ high) is the costly kind of mistake.

In [ ]:
fig, axes = plt.subplots(1, len(gases), figsize=(15, 3.4))
for ax, gas in zip(axes, gases, strict=True):
    proba, truth = pooled("discrete_bn", gas)
    seen = truth.notna()
    confusion = pd.DataFrame(score(truth[seen], proba[seen])["confusion"]).rename_axis(index="true", columns="pred")
    confusion = confusion.reindex(index=range(n_bins), columns=range(n_bins), fill_value=0)
    sns.heatmap(confusion.div(confusion.sum(axis=1), axis=0), annot=True, fmt=".2f", cmap="Blues",
                vmin=0, vmax=1, cbar=False, ax=ax)
    ax.set_title(f"{short(gas)} (n={int(seen.sum())})")
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "confusion.png"), bbox_inches="tight")
plt.show()

### 8.3 Do the learned CPTs match the study?

If S4 ← CO is real, then at a **fixed** NO2 bin the expected S4 bin should still change with CO. Each cell below is E[sensor bin | the two shown parents], read off the fitted CPT and averaged over the other parents (NOx, T, AH for S4) using their training distribution. Rows change CO; columns change the other gas.

The numbers under each heatmap summarize it: the mean spread (max − min) of E[S] along one parent while the other is held fixed. If the study is right, the CO spread is clearly non-zero for both sensors.

In [ ]:
def implied_sensor_mean(bn, sensor, row_parent, col_parent, train):
    """E[sensor bin | row_parent, col_parent], averaging the other parents over
    their joint training distribution (not their distribution given the two shown)."""
    table = bn.cpts[sensor].table
    parents = list(bn.network.structure[sensor])
    e = table @ np.arange(table.shape[-1])  # E[bin | all parents]
    e = np.moveaxis(e, [parents.index(row_parent), parents.index(col_parent)], [0, 1])
    others = [p for p in parents if p not in (row_parent, col_parent)]
    obs = train[others].dropna().astype(int).to_numpy()
    avg = e[(slice(None), slice(None), *obs.T)].mean(axis=-1)
    return pd.DataFrame(
        avg,
        index=pd.Index(range(avg.shape[0]), name=f"{short(row_parent)} bin"),
        columns=pd.Index(range(avg.shape[1]), name=f"{short(col_parent)} bin"),
    )


bn = fits[("nominal+cross", config.seed)]
panels = [(Sensor.S4, Gas.CO, Gas.NO2), (Sensor.S5, Gas.CO, Gas.NOX)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (sensor, row_p, col_p) in zip(axes, panels, strict=True):
    m = implied_sensor_mean(bn, sensor, row_p, col_p, splits[config.seed]["train"])
    spread_row = (m.max(axis=0) - m.min(axis=0)).mean()  # along row_p, col_p fixed
    spread_col = (m.max(axis=1) - m.min(axis=1)).mean()  # along col_p, row_p fixed
    sns.heatmap(m, annot=True, fmt=".2f", cmap="viridis", ax=ax, cbar_kws={"label": f"E[{short(sensor)} bin]"})
    ax.set_title(f"E[{short(sensor)} | {short(row_p)}, {short(col_p)}]  (nominal+cross, seed {config.seed})")
    ax.text(0, -0.25, f"spread along {short(row_p)}: {spread_row:.2f} bins; along {short(col_p)}: {spread_col:.2f} bins",
            transform=ax.transAxes, fontsize=9)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "cpt_check.png"), bbox_inches="tight")
plt.show()

**Reading:** **The learned CPTs match the cross-sensitivity findings.**
- **S4 behaves as a CO sensor.** At a fixed NO2 bin, E[S4] rises from ~0.68 (CO low) to ~1.1–1.4 (CO high): a spread of 0.61 bins along CO, against 0.13 along NO2 at fixed CO. So keeping S4 ← CO is right, and it's why NO2 is the weakest gas above.
- **S5 responds to both CO (spread 0.91 bins) and NOx (0.77 bins)**, so both of its edges are used.

## 9. Follow-ups at the chosen configuration

### 9.1 Drop-one-sensor ablation

In Phase 1b, every sensor was a child of CO only, so the BN treated all five sensors as **independent witnesses** of CO and became overconfident: removing any one of them *raised* the log-likelihood. Phase 2 gives S2–S5 their own gas parents, so they should stop double-counting.

For each gas, drop one sensor from the evidence and record the change from all sensors (Δ = without − with). A **positive Δ mean_log_likelihood** when a sensor is removed is the overconfidence symptom. Phase 1b is refit here on the same splits, bins and weather (T + AH), so the network is the only difference.

This cell predicts 20 extra (gas, evidence) pairs per seed.

In [ ]:
rows = []
for seed in seeds:
    sp = splits[seed]
    bn = fits[(chosen, seed)]
    bn_1b = fit_bn(sp, phase1b_structure())
    for s in (None, *sensors):
        label = "all sensors" if s is None else f"without {short(s)}"
        ev = tuple(e for e in evidence if e != s)
        for gas in gases:
            if s is None:
                proba = posteriors[(chosen, seed, "discrete_bn")][gas]  # already computed
            else:
                proba = bn.predict_proba(sp["val"], gas, ev)
            rows.append({"model": "Phase 2", "seed": seed, "gas": short(gas), "evidence": label,
                         **gas_metrics(proba, sp, gas)})
        proba = bn_1b.predict_proba(sp["val"], Gas.CO, ev)
        rows.append({"model": "Phase 1b", "seed": seed, "gas": "CO", "evidence": label,
                     **gas_metrics(proba, sp, Gas.CO)})
    print(f"seed {seed} done", flush=True)

abl = pd.DataFrame(rows)
abl.to_csv(config.paths.figure(FIG, "ablation.csv"), index=False)

ref = abl[abl.evidence == "all sensors"].set_index(["model", "seed", "gas"])
abl_delta = abl[abl.evidence != "all sensors"].copy()
for m in [*quality, "ECE", "MAE"]:
    abl_delta[f"Δ{m}"] = abl_delta[m].to_numpy() - ref.loc[
        list(zip(abl_delta.model, abl_delta.seed, abl_delta.gas, strict=True)), m
    ].to_numpy()
mean_std(abl_delta, ["model", "gas", "evidence"], ["Δaccuracy", "Δmean_log_likelihood", "ΔECE", "ΔMAE"])

In [ ]:
plot = abl_delta.assign(who=np.where(abl_delta.model == "Phase 1b", "CO (Phase 1b)", abl_delta.gas))
hue_order = ["CO (Phase 1b)", *map(short, gases)]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, m in zip(axes, ["Δmean_log_likelihood", "Δaccuracy"], strict=True):
    sns.barplot(data=plot, x="evidence", y=m, hue="who", hue_order=hue_order, errorbar="sd", ax=ax,
                legend=ax is axes[-1])
    ax.axhline(0, c="black", lw=0.8)
    ax.set_xlabel("")
    ax.set_ylabel(f"{m} (without − with)")
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle(f"Drop-one-sensor ablation on val ({chosen}, {n_bins} bins). "
             "Δmean_log_likelihood > 0 means the model was better off without that sensor.", y=1.03)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "ablation.png"), bbox_inches="tight")
plt.show()

**Reading:** **The Phase 1b overconfidence is reproduced, and Phase 2 removes it.**
- **Phase 1b:** removing *any* sensor raises CO's mean log-likelihood, by +0.02 to +0.10. The worst cases are S5 (+0.100 ± 0.010) and S1 (+0.089 ± 0.014): the five sensors were double-counted as independent CO witnesses.
- **Phase 2:** no removal raises log-likelihood beyond noise. The largest is +0.008 ± 0.011 (CO without S3). Each gas leans on the sensors wired to it: CO on S1 (−0.055 ± 0.018), NOx on S3 (−0.141 ± 0.025), then S4 and S5. NO2 leans on S4 (−0.033 ± 0.008), and NMHC on S2 (accuracy −0.114 ± 0.137, very noisy at n ≈ 180).
- Some overconfidence remains: CO's ECE is 0.074 and falls when S1, S4 or S5 is dropped (A2 check below).

### 9.2 EM vs complete-case

**Complete-case** counts each CPT only on the hours where the node and all its parents are observed. That is what `DiscreteBN` starts EM from, so it is `max_iter=0`. (`max_iter=1` would already be one E-step plus one M-step past it, i.e. one round of EM.) **EM** (`max_iter=100`, the grid's fit) also uses the hours with a missing reference, weighting each possible gas level by its posterior.

First, how many training hours complete-case loses per CPT:

In [ ]:
train = splits[config.seed]["train"]
lost = {
    short(node): train[[*parents, node]].isna().any(axis=1).mean()
    for node, parents in structure(edge_sets[chosen]).items()
}
pd.Series(lost, name="fraction of train hours dropped by complete-case").round(3).to_frame().T

In [ ]:
rows = []
for seed in seeds:
    sp = splits[seed]
    em_fits = {
        "EM": fits[(chosen, seed)],
        "complete-case": fit_bn(sp, structure(edge_sets[chosen]), max_iter=0),
    }
    for label, bn in em_fits.items():
        heldout_ll = bn.log_likelihood(sp["val"]) / len(sp["val"])  # log P(observed cells | inputs), per hour
        for gas in gases:
            if label == "EM":
                proba = posteriors[(chosen, seed, "discrete_bn")][gas]
            else:
                proba = bn.predict_proba(sp["val"], gas, evidence)
            rows.append({"fit": label, "seed": seed, "gas": short(gas), "heldout_ll": heldout_ll,
                         **gas_metrics(proba, sp, gas)})
    print(f"seed {seed} done", flush=True)

em = pd.DataFrame(rows)
em.to_csv(config.paths.figure(FIG, "em_vs_complete_case.csv"), index=False)
mean_std(em, ["gas", "fit"], [*quality, "ECE", "MAE", "heldout_ll"])

In [ ]:
# paired: EM − complete-case on the same seed (positive = EM better, except ECE / MAE)
em_cols = [*quality, "ECE", "MAE", "heldout_ll"]
em_idx = em.set_index(["fit", "seed", "gas"])
em_diff = (em_idx.loc["EM", em_cols] - em_idx.loc["complete-case", em_cols]).reset_index()
mean_std(em_diff, ["gas"], em_cols, digits=4)

**Reading:** **EM does not help prediction here, including for NMHC.**
- **Joint fit:** EM's held-out joint log-likelihood is higher than complete-case by +0.116 ± 0.061 nats/hour, mostly because it now uses the ~86% of hours without NMHC.
- **Per gas, EM is slightly worse than complete-case:**
  - CO: log-lik −0.010 ± 0.005.
  - NOx: accuracy −0.012 ± 0.003, log-lik −0.025 ± 0.002.
  - NO2: log-lik −0.013 ± 0.004.
  - **NMHC, the gas EM was meant to help, is worse by the most:** accuracy −0.041 ± 0.019, log-lik −0.071 ± 0.035, MAE +10.6 ± 3.1 µg/m³.
- **Why, most likely:** On hours with no NMHC reading, EM fills NMHC in from S2 and the hour, using a model that treats the gases as independent given the hour (A1, violated below). For NMHC those filled-in hours are ~86% of the training counts behind S2's CPT. They also cover months with no NMHC reading at all: every reading is from March–April 2004 (section 4). So EM fits S2 to what the model expects rather than to measured NMHC.
- **For the report:** using the hours with a missing reference improves the joint fit but not per-gas prediction. For NMHC, complete-case on the hours with a real reading is the better estimator. Phase 3's latent-gas EM should be judged against complete-case, not assumed to help.

## 10. Assumptions A1–A4

### A1: are the gases independent given the hour?

Two checks, on the hours where both gases are observed:
1. **Partial Spearman ρ** of each gas pair, controlling for the hour (24 dummies), on train. Near 0 would support A1.
2. **In the BN's own terms:** held-out log-likelihood gain per hour of P(gᵢ | gⱼ, hour) over P(gᵢ | hour), fit on train and scored on val, at the chosen bins. A clearly positive gain means gⱼ still tells us about gᵢ within the hour, so A1 is violated.

In [ ]:
rows = []
for seed in seeds:
    sp = splits[seed]
    for gi in gases:
        for gj in gases:
            if gi == gj:
                continue
            rho, _ = partial_spearman(sp["train_raw"], gi, gj, [Time.HOUR], categorical=[Time.HOUR])
            cols = [Time.HOUR, gi, gj]
            train_ij, val_ij = sp["train"][cols].dropna(), sp["val"][cols].dropna()
            if val_ij.empty:
                continue
            base = DiscreteBN(Network(structure={gi: (Time.HOUR,)}, cards=sp["cards"])).fit(train_ij)
            full = DiscreteBN(Network(structure={gi: (Time.HOUR, gj)}, cards=sp["cards"])).fit(train_ij)
            gain = (full.log_likelihood(val_ij) - base.log_likelihood(val_ij)) / len(val_ij)
            rows.append({"seed": seed, "gas": short(gi), "given": short(gj),
                         "partial_rho": rho, "ll_gain": gain, "n": len(val_ij)})

a1 = pd.DataFrame(rows)
a1.to_csv(config.paths.figure(FIG, "a1_gas_given_gas.csv"), index=False)

names = [short(g) for g in gases]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (col, title, cmap, center) in zip(axes, [
    ("partial_rho", "partial Spearman ρ | hour (train)", "RdBu_r", 0),
    ("ll_gain", "val log-lik gain/hour of P(gas | given, hour)", "Reds", None),
], strict=True):
    m = a1.groupby(["gas", "given"])[col].mean().unstack().reindex(index=names, columns=names)
    s = a1.groupby(["gas", "given"])[col].std().unstack().reindex(index=names, columns=names)
    annot = (m.round(2).astype(str) + "\n±" + s.round(2).astype(str)).where(m.notna(), "")
    sns.heatmap(m, annot=annot.values, fmt="", cmap=cmap, center=center, ax=ax,
                vmin=-1 if center == 0 else None, vmax=1 if center == 0 else None)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "a1_gases_given_hour.png"), bbox_inches="tight")
plt.show()

**Reading:** **A1 is not supported.** Within an hour, the gases still co-move strongly:
- Partial ρ | hour is 0.64–0.87 (CO–NMHC 0.87, NO2–NOx 0.79, NMHC–NOx 0.78, CO–NOx 0.72; NMHC pairs on n ≈ 175 val hours).
- Adding one gas as a parent of another improves held-out log-likelihood by up to 0.28 nats/hour (NO2 | NOx), 0.18 (CO | NMHC) and 0.16 (CO | NOx).
- Traffic drives all of them hour by hour, not just through the daily cycle.

This is the main structural limitation of Phase 2. Evidence about one gas reaches another only through a shared sensor child (S4, S5), which is why CO lost S2's evidence. Candidate fixes for Phase 3:
- a shared latent "traffic" parent above the gases, or
- gas–gas edges (e.g. CO → NOx → NO2).

### A2: are the sensors independent given their gas parents and the weather?

Partial Spearman ρ between every pair of sensors, controlling for **all four gases**, T, AH and hour (train, seed 0, hours with every gas observed). The BN's A2 is a stronger claim (each sensor depends only on *its own* gas parents), so controlling for all gases is generous to it: anything left is dependence the BN can't represent. Large values mean the sensors share something the network doesn't model (common drift, a shared heater, an unmodelled gas such as O3), which double-counts their evidence.

In [ ]:
train_raw = splits[config.seed]["train_raw"]
controls = [*gases, *weather, Time.HOUR]
a2 = pd.DataFrame(np.nan, index=[short(s) for s in sensors], columns=[short(s) for s in sensors])
for i, si in enumerate(sensors):
    for sj in sensors[i + 1:]:
        rho, n_a2 = partial_spearman(train_raw, si, sj, controls, categorical=[Time.HOUR])
        a2.loc[short(si), short(sj)] = a2.loc[short(sj), short(si)] = rho
a2.to_csv(config.paths.figure(FIG, "a2_sensor_partial_rho.csv"))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(a2, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax)
ax.set_title(f"Sensor–sensor partial ρ | all gases, T, AH, hour (n={n_a2})")
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "a2_sensor_residuals.png"), bbox_inches="tight")
plt.show()

**Reading:** **A2 is not supported: the sensors stay strongly dependent after controlling for every gas.** This check uses only the 582 seed-0 train hours with all four gases observed, so it covers March–April 2004 only.
- The largest residual correlations are S2–S3 −0.69, S2–S4 0.65, S3–S5 −0.59, S3–S4 −0.55, S1–S5 0.50 and S2–S5 0.48.
- The large S2 values support the team's caveat. S2 shares a lot with S3, S4 and S5 beyond NMHC, so it's measuring traffic pollution broadly, and **NMHC predictions may partly reflect traffic in general rather than NMHC specifically.**
- S1, S4 and S5 are all CO witnesses and partly double-count, which is the most likely source of CO's remaining overconfidence (ECE 0.074; the curve sits below the diagonal at 0.6–0.85 confidence).

### A3: is missingness ignorable?

MAR can't be proven from the data, but two things make it plausible or not:
- **When** the references are missing. Long contiguous gaps point to analyzer downtime (unrelated to the gas level); scattered gaps at high or low readings would not.
- **Whether the sensors look different** on hours with a missing reference. Under MCAR the nominal sensor's distribution is the same either way; a shift means missingness depends on the gas level, which is fine for MAR only if the shift is explained by the observed sensors, hour and weather.

In [ ]:
missing = df[list(gases)].isna().rename(columns=short)
fig, ax = plt.subplots(figsize=(10, 3))
missing.resample("W").mean().plot(ax=ax)
ax.set_ylabel("fraction missing (weekly)")
ax.set_xlabel("")
ax.set_title("Reference missingness over time")
plt.tight_layout()
plt.savefig(config.paths.figure(FIG, "a3_missingness_over_time.png"), bbox_inches="tight")
plt.show()

# median of each gas's nominal sensor, on hours with vs without that gas's reference
pd.DataFrame({
    f"{short(g)} via {short(s)}": {
        "median when reference observed": df.loc[df[g].notna(), s].median(),
        "median when reference missing": df.loc[df[g].isna(), s].median(),
        "hours missing": int(df[g].isna().sum()),
    }
    for s, (g,) in nominal.items()
}).round(1)

**Reading:**
- **CO, NOx and NO2: MAR is supported.** Their gaps are a ~5% background plus multi-week outages (April–May, August–October, early January), with NOx and NO2 dropping out together (one analyzer). That looks like analyzer downtime. The nominal sensors read only somewhat differently during gaps (S1 median 1015 vs 1075, S3 869 vs 792, S4 unchanged).
- **NMHC is missing:** it is observed only in March–April 2004 and missing everywhere after (section 4). S2's median barely differs (907 vs 927), but the missingness is determined by the date, so it depends on season. Hour and weather are observed, so this is still MAR in principle. In practice, everything learned about NMHC comes from spring conditions.

### A4: are hours independent?

Lag-1 autocorrelation of the concentration residuals (estimate − reference) on consecutive val hours, chosen configuration. Near 0 supports A4; a large value means the next hour's error is predictable from this one's, which is exactly what Phase 4's HMM would use.

In [ ]:
def lag1_autocorr(resid):
    r = resid.dropna()
    consecutive = r.index.to_series().diff() == pd.Timedelta("1h")
    return r[consecutive].corr(r.shift(1)[consecutive])


rows = []
for seed in seeds:
    sp = splits[seed]
    for gas in gases:
        proba = posteriors[(chosen, seed, "discrete_bn")][gas]
        c_bar = sp["c_bar"][gas]
        estimate = pd.Series(proba.to_numpy() @ c_bar.loc[proba.columns].to_numpy(), index=proba.index)
        rows.append({"seed": seed, "gas": short(gas),
                     "lag1_autocorr": lag1_autocorr(estimate - sp["val_raw"][gas])})
a4 = pd.DataFrame(rows)
mean_std(a4, ["gas"], ["lag1_autocorr"])

**Reading:** **A4 is not supported.** The lag-1 autocorrelation of the concentration residuals is 0.64 (CO), 0.45 (NMHC), 0.74 (NOx) and 0.89 (NO2):
- When the model is off in one hour, it's off the same way the next hour, so neighbouring hours carry information the static network ignores. That's the case for Phase 4's HMM.
- It also means the effective number of independent hours is much smaller than `n`. Seed-to-seed std is the better guide to uncertainty.

## 11. Drift check: time-based split

The blocked split mixes weeks from the whole year into train and val. A chronological split (train = first half of the data, val = the next quarter) asks whether CPTs learned on earlier months still hold later. The sensors are known to drift, so some loss is expected. Chosen configuration; one run, since the split has no seed. Compare with the blocked-split val numbers.

In [ ]:
ts = make_split(config.seed, split=time_split)
print({k: f"{ts[k].index.min():%Y-%m-%d} → {ts[k].index.max():%Y-%m-%d}" for k in ("train_raw", "val_raw", "test_raw")})
bn_ts = fit_bn(ts, structure(edge_sets[chosen]))
hour_ts = HourOnlyBaseline(cards=ts["cards"]).fit(ts["train"])

scorable = [g for g in gases if ts["val"][g].notna().any()]  # a gas with no val readings can't be scored
print("scored on the time split:", [short(g) for g in scorable])
rows = []
for model in (bn_ts, hour_ts):
    rows += evaluate_all(model, ts, gases=scorable)[1]
drift = pd.DataFrame(rows).assign(split="time")

blocked = (grid[grid.edges == chosen]
           .groupby(["gas", "model"], sort=False)[["n", *quality, "ECE", "MAE"]].mean().reset_index()
           .assign(split="blocked (mean over seeds)"))
drift_table = pd.concat([blocked, drift[blocked.columns]]).set_index(["gas", "model", "split"]).sort_index()
drift_table.to_csv(config.paths.figure(FIG, "drift_time_split.csv"))
drift_table.round(3)

**Reading:** **Clear drift.** Trained on March–September 2004 and scored on September–December 2004. NMHC can't be scored here: it has no readings after April 2004.
- **CO:** accuracy 0.773 → 0.719, mean log-lik −0.570 → −0.807, MAE 0.544 → 0.866 mg/m³.
- **NOx:** accuracy 0.709 → 0.655, log-lik −0.633 → −1.071, ECE 0.060 → 0.148. MAE more than doubles (89.5 → 219.7 ppb), because winter NOx is far above anything in the summer training bins, and the estimate can't exceed the top bin's summer mean.
- **NO2:** the BN falls *below* the hour baseline (accuracy 0.497 vs 0.554, log-lik −1.050 vs −0.975).
- The hour baseline degrades too, so part of this is a seasonal shift in gas levels rather than only sensor drift.
- The BN still beats the baseline for CO and NOx, but the blocked-split numbers are optimistic for later months.

## 12. Test set: chosen configuration, reported once

Everything above was chosen on val. This is the only cell that reads **test**. Same fitted models as the grid (trained on train only), same metrics.

In [ ]:
rows = []
for seed in seeds:
    sp = splits[seed]
    hour = HourOnlyBaseline(cards=sp["cards"]).fit(sp["train"])
    for model in (fits[(chosen, seed)], hour):
        rows += [{"seed": seed, **r} for r in evaluate_all(model, sp, part="test")[1]]
test_results = pd.DataFrame(rows)
test_results.to_csv(config.paths.figure(FIG, "test_chosen.csv"), index=False)
mean_std(test_results, ["gas", "model"], [*quality, "ECE", "MAE", "floor_MAE", "RMSE"])

## Summary

Numbers are val, 3 bins, mean ± std over seeds 0–2, unless marked test. NMHC is scored only on hours with a real reading.

- **Per gas vs baseline:** 
  - CO and NOx beat `HourOnlyBaseline` clearly. CO: accuracy 0.773 vs 0.586 (n = 1853), MAE 0.544 vs 0.857 mg/m³. NOx: accuracy 0.709 vs 0.495 (n = 1855), MAE 89.5 vs 136.5 ppb. 
  - NMHC beats it on average with a wide spread: accuracy 0.732 ± 0.113 vs 0.613, n ≈ 180 per seed. 
  - NO2 beats it only with the cross edges, and only slightly (log-lik +0.034 ± 0.009), because S4 is mostly a CO sensor.

- **CO vs Phase 1:** 
  - lower accuracy (0.776 vs 0.828, seed 0) with better log-likelihood (−0.541 vs −0.662), as anticipated. S2 is tied only to NMHC, so it no longer helps CO.

- **Nominal-only vs nominal + cross:** 
  - the cross edges improve accuracy, log-likelihood and MAE for CO, NOx and NO2 (CO MAE −0.046 ± 0.019, NOx −5.8 ± 3.2 ppb). Calibration is mixed: ECE is worse for CO and NOx and better for NO2.

- **Drop-one-sensor ablation:** 
  - Phase 1b reproduces the overconfidence (removing any sensor raises log-lik by +0.02 to +0.10). Phase 2 removes it: no removal raises log-lik beyond noise.

- **EM vs complete-case:** 
  - EM improves the held-out joint log-lik (+0.116 ± 0.061) but is slightly worse per gas. NMHC is worst: accuracy −0.041 ± 0.019, MAE +10.6 ± 3.1 µg/m³.

- **Chosen configuration:** 
  - nominal + cross at 3 bins. It has the lowest MAE relative to the baseline over CO, NOx and NO2: 0.754 ± 0.027 vs 0.796 ± 0.023.

- **Test** (chosen configuration, reported once):
  - CO: accuracy 0.778 ± 0.012 vs 0.580, log-lik −0.569 vs −0.898, MAE 0.61 vs 0.94 mg/m³ (n = 1751).
  - NMHC: accuracy 0.828 ± 0.021 vs 0.643, MAE 97 ± 24 vs 129 µg/m³ (n ≈ 124).
  - NOx: accuracy 0.735 vs 0.505, MAE 99 ± 17 vs 144 ppb (n = 1776).
  - NO2: accuracy 0.572 vs 0.556, MAE 30.1 vs 31.1 µg/m³ (n = 1775).

- **Drift:** 
  - on a chronological split, CO, NOx and NO2 all degrade. CO MAE goes 0.544 → 0.866, NOx log-lik −0.63 → −1.07, and NO2 falls below the hour baseline. NMHC has no readings after April 2004, so it can't be checked.

- **CPTs vs cross-sensitivity:** 
  - they match. At fixed NO2, S4 moves 0.61 bins with CO and only 0.13 with NO2. S5 responds to both CO and NOx.

- **Assumptions:**
  - **A1 not supported:** gases co-move within the hour, with partial ρ 0.64–0.87 and log-lik gains up to 0.28 nats/hour.
  - **A2 not supported:** sensor residuals stay correlated up to |0.69| given all gases, S2 most of all.
  - **A3 supported:** for CO, NOx and NO2 (analyzer outages). NMHC readings cover only March–April 2004.
  - **A4 not supported:** lag-1 residual autocorrelation is 0.45–0.89.

- **Limitations:**
  - S2 tracks all the traffic gases about equally, so NMHC predictions may partly reflect traffic pollution in general rather than NMHC specifically; the A2 check supports this.
  - NMHC results rest on n ≈ 70–210 hours per split, all from spring 2004.
  - 3 bins only: CO and NOx MAE sit near the resolution floor, but finer bins would make the multi-parent sensor CPTs very sparse.
  - A1 and A4 are the biggest model errors. They point to a shared traffic/latent parent (Phase 3) and temporal edges (Phase 4).